# Ablation Study 4 — Edge Feature Contribution

**Question:** How much do bond-type features (single/double/triple/aromatic one-hot) contribute to GATv2's performance?

**Why GATv2 only?**  
- GCN (`GCNConv`): never uses `edge_attr` — already the no-edge baseline  
- GAT (`GATConv`): never uses `edge_attr` in this implementation — already no-edge  
- GATv2 (`GATv2Conv`): actively uses `edge_dim=4` — bond types enter the attention score inside LeakyReLU  

So the clean experiment is: GCN (no edges) vs GAT (no edges) vs GATv2-no-edges vs GATv2-with-edges.

This tells you:
- If GATv2+edges >> GATv2-no-edges: bond types are genuinely informative
- If GATv2-no-edges ≈ GATv2+edges but >> GAT: dynamic attention (not edge features) drives the gain
- If GATv2-no-edges ≈ GAT: dynamic attention alone adds little without edge info

**Output directory:** `MyDrive/Ablation/Study4_EdgeFeatureSelection/`

In [ ]:
# ── Cell 1: Mount Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────────
import torch, subprocess, sys
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cu121' if torch.cuda.is_available() else 'cpu'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-scatter', 'torch-sparse',
                '-f', f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric'], check=True)
print('PyG installed.')

In [ ]:
# ── Cell 3: Clone repo & set paths ────────────────────────────────────────────
import os, sys, subprocess

REPO_URL  = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # ← update this
REPO_DIR  = '/content/gnn_project'
DRIVE_OUT = '/content/drive/MyDrive/Ablation/Study4_EdgeFeatureSelection'
DATA_ROOT = '/content/qm9_data'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

for sub in ['checkpoints', 'logs', 'results', 'plots']:
    os.makedirs(f'{DRIVE_OUT}/{sub}', exist_ok=True)

print(f'Output root: {DRIVE_OUT}')

In [ ]:
# ── Cell 4: USER CONFIG ───────────────────────────────────────────────────────

# ---------- Training ----------
EPOCHS      = 300
PATIENCE    = 30
BATCH_SIZE  = 128
LR          = 5e-4

# ---------- Model (fixed) ----------
HIDDEN_DIM   = 256
NUM_LAYERS   = 6
DROPOUT      = 0.0
HEADS        = 8
FEATURE_MODE = 'topology'

# ---------- Runs ----------
# Each entry: (model_name, use_edge_attr, run_label)
# GCN and GAT are included as reference baselines (no edge_attr by design)
RUNS = [
    ('gcn',   False, 'GCN (no edges)'),
    ('gat',   False, 'GAT (no edges)'),
    ('gatv2', False, 'GATv2 (no edges)'),
    ('gatv2', True,  'GATv2 (with edges)'),
]

# ---------- Dataset ----------
TARGET_IDX  = 7
SEED        = 42
SPLIT       = [0.8, 0.1, 0.1]

print(f'Runs planned: {len(RUNS)}')
for r in RUNS:
    print(f'  {r[2]}')

In [ ]:
# ── Cell 5: Load data ─────────────────────────────────────────────────────────
from data.loader import get_dataloaders

base_cfg = {
    'dataset':  {'target': TARGET_IDX, 'split': SPLIT, 'seed': SEED, 'feature_mode': FEATURE_MODE},
    'training': {'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'patience': PATIENCE, 'lr': LR},
}
train_loader, val_loader, test_loader, normalizer = get_dataloaders(base_cfg, root=DATA_ROOT)
print('Data loaded.')

In [ ]:
# ── Cell 6: Training functions ────────────────────────────────────────────────
import copy, csv, torch
import torch.nn.functional as F
from tqdm import tqdm
from data.features import select_features, get_feature_dims
from models import build_model
from models.gatv2 import GATv2  # direct import to allow edge_dim override

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

def mae(pred, target):
    return (pred - target).abs().mean().item()


def run_epoch(model, loader, optimizer, device, normalizer, feature_mode, model_name, use_edge_attr, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_targets = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            batch = select_features(batch, mode=feature_mode)
            batch = batch.to(device)
            if model_name == 'gatv2' and use_edge_attr:
                pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            elif model_name == 'gatv2' and not use_edge_attr:
                # GATv2 built with edge_dim=None — pass no edge_attr
                pred = model(batch.x, batch.edge_index, None, batch.batch)
            else:
                pred = model(batch.x, batch.edge_index, batch.batch)
            target = batch.y.view(-1)
            loss = F.mse_loss(pred, target)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * batch.num_graphs
            all_preds.append(normalizer.denormalize(pred.detach().cpu()))
            all_targets.append(normalizer.denormalize(target.detach().cpu()))
    n = sum(t.size(0) for t in all_targets)
    return total_loss / n, mae(torch.cat(all_preds), torch.cat(all_targets))


def build_run_model(model_name, use_edge_attr, feature_dims):
    """Construct the model for this run, handling the GATv2 edge_dim flag."""
    cfg = {
        'dataset':  {'target': TARGET_IDX, 'split': SPLIT, 'seed': SEED, 'feature_mode': FEATURE_MODE},
        'training': {'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'patience': PATIENCE, 'lr': LR},
        'model':    {'hidden_dim': HIDDEN_DIM, 'num_layers': NUM_LAYERS, 'dropout': DROPOUT, 'heads': HEADS}
    }
    if model_name == 'gatv2':
        # Build GATv2 directly with explicit edge_dim control
        edge_dim = 4 if use_edge_attr else None
        return GATv2(
            node_dim=feature_dims['node_dim'],
            hidden_dim=HIDDEN_DIM,
            num_layers=NUM_LAYERS,
            heads=HEADS,
            edge_dim=edge_dim,
            dropout=DROPOUT,
            share_weights=False,
        )
    return build_model(model_name, cfg, feature_dims=feature_dims)


def train_run(model_name, use_edge_attr, run_label, run_id):
    feature_dims = get_feature_dims(FEATURE_MODE)
    model = build_run_model(model_name, use_edge_attr, feature_dims).to(DEVICE)
    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  [{run_id}] {run_label} | params={param_count:,}')

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    ckpt_path = f'{DRIVE_OUT}/checkpoints/{run_id}_best.pt'
    log_path  = f'{DRIVE_OUT}/logs/{run_id}_history.csv'

    best_val_mae, best_state, patience_ctr = float('inf'), None, 0
    history = []

    with open(log_path, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=['epoch','train_loss','val_loss','val_mae','lr']).writeheader()

    pbar = tqdm(range(1, EPOCHS + 1), desc=run_id, leave=True)
    for epoch in pbar:
        tr_loss, _ = run_epoch(model, train_loader, optimizer, DEVICE, normalizer,
                               FEATURE_MODE, model_name, use_edge_attr, train=True)
        vl_loss, vl_mae_val = run_epoch(model, val_loader, None, DEVICE, normalizer,
                                        FEATURE_MODE, model_name, use_edge_attr, train=False)
        lr_now = optimizer.param_groups[0]['lr']
        row = {'epoch': epoch, 'train_loss': f'{tr_loss:.6f}', 'val_loss': f'{vl_loss:.6f}',
               'val_mae': f'{vl_mae_val:.6f}', 'lr': f'{lr_now:.2e}'}
        history.append(row)
        with open(log_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=['epoch','train_loss','val_loss','val_mae','lr']).writerow(row)
        pbar.set_postfix(val_mae=f'{vl_mae_val:.4f}')
        if vl_mae_val < best_val_mae:
            best_val_mae = vl_mae_val
            best_state   = copy.deepcopy(model.state_dict())
            patience_ctr = 0
            torch.save(best_state, ckpt_path)
        else:
            patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'  Early stop at epoch {epoch}')
            break

    print(f'  [{run_id}] best val MAE = {best_val_mae:.4f} Ha')
    return {'run_id': run_id, 'label': run_label, 'model': model_name,
            'use_edge_attr': use_edge_attr, 'best_val_mae': best_val_mae,
            'params': param_count, 'checkpoint': ckpt_path}


print('Functions defined.')

In [ ]:
# ── Cell 7: Run all 4 variants ────────────────────────────────────────────────
import pandas as pd

results = []
results_path = f'{DRIVE_OUT}/results/study4_results.csv'

for i, (model_name, use_edge_attr, run_label) in enumerate(RUNS):
    run_id = run_label.lower().replace(' ', '_').replace('(', '').replace(')', '')
    print(f'\n[{i+1}/{len(RUNS)}] Running {run_label} ...')
    result = train_run(model_name, use_edge_attr, run_label, run_id)
    results.append(result)
    pd.DataFrame(results).to_csv(results_path, index=False)
    print(f'  Saved → {results_path}')

df = pd.DataFrame(results)
print('\n── Study 4 Results ──')
print(df[['label', 'best_val_mae', 'params']].to_string(index=False))

In [ ]:
# ── Cell 8: CVPR-style bar chart ──────────────────────────────────────────────
import matplotlib, matplotlib.pyplot as plt
import numpy as np, pandas as pd

matplotlib.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 9, 'axes.titlesize': 9, 'axes.labelsize': 9,
    'xtick.labelsize': 7, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'figure.dpi': 300, 'axes.spines.top': False, 'axes.spines.right': False,
})

# IBM palette — ordered to group model families visually
COLORS = ['#648FFF', '#785EF0', '#DC267F', '#FE6100']

df = pd.read_csv(f'{DRIVE_OUT}/results/study4_results.csv')
labels = df['label'].tolist()
maes   = df['best_val_mae'].tolist()

fig, ax = plt.subplots(figsize=(3.5, 2.6))
x = np.arange(len(labels))
bars = ax.bar(x, maes, color=COLORS[:len(labels)], edgecolor='white', linewidth=0.5, width=0.55)

for bar, v in zip(bars, maes):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
            f'{v:.3f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylabel('Val MAE (Ha)')
ax.set_title('Edge Feature Contribution to U0 Prediction')
fig.tight_layout(pad=0.4)

plot_path = f'{DRIVE_OUT}/plots/study4_edge_features.pdf'
fig.savefig(plot_path, format='pdf', bbox_inches='tight')
fig.savefig(plot_path.replace('.pdf', '.png'), format='png', bbox_inches='tight', dpi=300)
plt.show()
print(f'Plot saved → {plot_path}')